# Helper

In [152]:
import zipfile
from pathlib import Path
from io import BytesIO  # Import BytesIO
from PyPDF2 import PdfReader  # No PdfReadError in the latest PyPDF2
import re

In [153]:

#--------------------------------------- Text Extraction -------------------------------------------#

def process_file(file_path):
    # Get the extension of the submitted file
    file_extension = Path(file_path).suffix

    if file_extension == '.zip':
        # Open the zip file and process its contents
        file_contents = {}
        with zipfile.ZipFile(file_path, 'r') as zip_ref:
            for file_name in zip_ref.namelist():
                # Skip macOS metadata files
                if file_name.startswith('__MACOSX/') or file_name.endswith('/'):
                    print(f"Skipping unwanted file: {file_name}")
                    continue

                if file_name.endswith('.pdf') or file_name.endswith('.py'):
                    try:
                        with zip_ref.open(file_name) as file:
                            # Read as binary content for compatibility
                            content = store_file(file.read(), file_name)
                            file_contents[file_name] = content
                    except Exception as e:
                        print(f"Error processing {file_name}: {e}")
        return file_contents

    elif file_extension in ['.pdf', '.py']:
        # If it's a .pdf or .py file, store its content directly
        with open(file_path, 'rb') as file:
            content = store_file(file.read(), file_path)
        return {file_path: content}
    else:
        print("Unsupported file type.")
        return None


def store_file(file_content, file_name):
    # Check file extension to determine the handling method
    if file_name.endswith('.pdf'):
        return extract_pdf_text(file_content)
    elif file_name.endswith('.py'):
        return read_python_file(file_content)


def extract_pdf_text(file_content):
    # Read the content from the PDF file-like object
    try:
        reader = PdfReader(BytesIO(file_content))  # Use BytesIO for file-like object
        text = ''
        for page in reader.pages:
            text += page.extract_text() or ""  # Safely handle None
        print("Extracted text from PDF.")
        return text
    except Exception as e:  # Handle general exceptions
        print(f"Error reading PDF: {e}")
        return f"Error: Unable to read PDF."


def read_python_file(file_content):
    # Read the Python (.py) file content
    encodings = ['utf-8', 'ISO-8859-1', 'latin-1']
    for encoding in encodings:
        try:
            content = file_content.decode(encoding)
            print(f"Successfully read Python file with {encoding} encoding.")
            return content
        except UnicodeDecodeError:
            print(f"Failed to read Python file with {encoding} encoding. Trying next...")
    raise UnicodeDecodeError("Unable to read Python file with any encoding.")


def extract_text_between_intro_and_references(pdf_text):
    pdf_text_lower = pdf_text.lower()  # Normalize case for search

    # Step 1: Locate "Table of Tables" and find first "Introduction" after it
    table_tables_match = re.search(r"table of tables", pdf_text_lower)
    if table_tables_match:
        intro_after_tables = re.search(r"introduction", pdf_text_lower[table_tables_match.end():])
        if intro_after_tables:
            start_index = table_tables_match.end() + intro_after_tables.start()
        else:
            start_index = None  # No valid introduction found after Table of Tables

    # Step 2: If no intro found after Table of Tables, check "Table of Figures"
    if start_index is None:
        table_figures_match = re.search(r"table of figures", pdf_text_lower)
        if table_figures_match:
            intro_after_figures = re.search(r"introduction", pdf_text_lower[table_figures_match.end():])
            if intro_after_figures:
                start_index = table_figures_match.end() + intro_after_figures.start()
            else:
                start_index = None  # No valid introduction found after Table of Figures

    # Step 3: If no "Table of Tables" or "Table of Figures", take the first "Introduction"
    if start_index is None:
        intro_match = re.search(r"introduction", pdf_text_lower)
        if intro_match:
            start_index = intro_match.start()
        else:
            return "No valid Introduction section found."

    # Step 4: Find the last occurring "Reference" (search from the end)
    ref_matches = list(re.finditer(r"reference", pdf_text_lower))
    if not ref_matches:
        return "No Reference section found."

    end_index = ref_matches[-1].start()  # Last occurrence of "Reference"

    # Step 5: Extract and return text between "Introduction" and last "Reference"
    extracted_text = pdf_text[start_index:end_index].strip()
    
    return extracted_text



#--------------------------------------- Text Extraction -------------------------------------------#



In [154]:
import nltk
from nltk.tokenize import sent_tokenize

# Ensure nltk sentence tokenizer is available
nltk.download('punkt')



[nltk_data] Error loading punkt: <urlopen error [SSL:
[nltk_data]     CERTIFICATE_VERIFY_FAILED] certificate verify failed:
[nltk_data]     unable to get local issuer certificate (_ssl.c:1002)>


False

In [155]:
#--------------------------------------- Embeddings -------------------------------------------#

from sentence_transformers import SentenceTransformer

# Load pre-trained SBERT model
model = SentenceTransformer('all-MiniLM-L6-v2')

def get_sentences(text):
    # Remove extra newlines but keep paragraph breaks
    text = re.sub(r"\n+", " ", text)  # Replace multiple newlines with space
    
    # Remove extra spaces around punctuation
    text = re.sub(r'\s+([.,!?])', r'\1', text)

    # Fix broken words (e.g., "c hange" -> "change") using a basic pattern
    text = re.sub(r'\b(\w)\s+(\w)\b', r'\1\2', text)

    # Remove multiple spaces
    text = re.sub(r'\s+', ' ', text).strip()

    # Tokenize into proper sentences
    sentences = sent_tokenize(text)

    # # Remove punctuation from each sentence
    # sentences = [re.sub(r'[^\w\s]', '', sentence) for sentence in sentences]

    return list(dict.fromkeys(sentences))

    # text = text.split('.')  # Split by full stop
    # return list(dict.fromkeys([sen.strip() for sen in text[:-1]]))


def get_embeddings(sentences):
    embeddings = {}

    # Generate sentence embeddings
    sentence_embeddings = model.encode(sentences)

    # Display the embeddings
    for i, embedding in enumerate(sentence_embeddings):
        embeddings[sentences[i]] = embedding
    
    return embeddings


#--------------------------------------- Embeddings -------------------------------------------#


# Main

In [156]:
from sklearn.metrics.pairwise import cosine_similarity

In [157]:
# Load and process the input file from a manual link
pdf_path = "test_files/Test3.pdf" 
with open(pdf_path, "rb") as f:
    file_bytes = f.read()

In [158]:
# Process the input file
text = extract_pdf_text(file_bytes)
sentences = get_sentences(text)
embeddings_self = get_embeddings(sentences)

# Load existing files for comparison
file1 = process_file("test_files/Test1.pdf")
text1 = next(iter(file1.values()))
sentences1 = get_sentences(text1)
embeddings_other_1 = get_embeddings(sentences1)

file2 = process_file("test_files/Test2.pdf")
text2 = next(iter(file2.values()))
sentences2 = get_sentences(text2)
embeddings_other_2 = get_embeddings(sentences2)

existing_files = [embeddings_other_1, embeddings_other_2]

# Dictionary to store flagged sentences
flagged_sentences = {}


Extracted text from PDF.
Extracted text from PDF.
Extracted text from PDF.


In [159]:
# txt = extract_text_between_intro_and_references(text)
# txt

In [160]:
from pymilvus import CollectionSchema, FieldSchema, DataType, Collection, connections, utility

# Connect to the Milvus server
connections.connect("default", host="localhost", port="19530")

# Check connection
if connections.has_connection("default"):
    print("Connected to Milvus successfully!")
else:
    print("Failed to connect to Milvus.")

Connected to Milvus successfully!


In [161]:
# Define Fields
id_field = FieldSchema(name="id", dtype=DataType.INT64, is_primary=True, auto_id=True)
sentence_field = FieldSchema(name="sentence", dtype=DataType.VARCHAR, max_length=384)  # Adjust max_length as needed
embedding_field = FieldSchema(name="embedding", dtype=DataType.FLOAT_VECTOR, dim=384)  # Adjust dim as needed

# Define Schema
schema = CollectionSchema(fields=[id_field, sentence_field, embedding_field], description="Sentence Embeddings Collection")

collection_name = "sentence_embeddings"

# Drop the collection if it exists
existing_collections = utility.list_collections()  # Correct way to list collections
if collection_name in existing_collections:
    Collection(collection_name).drop()
    print(f"Existing collection '{collection_name}' dropped.")

# Now, recreate the collection
collection = Collection(name=collection_name, schema=schema)
collection.create_index(field_name="embedding", index_params={"index_type": "IVF_FLAT", "metric_type": "L2", "params": {"nlist": 128}})
print("New collection created successfully.")

Existing collection 'sentence_embeddings' dropped.
New collection created successfully.


In [162]:
# Inserting embeddings

import numpy as np

def insert_embeddings(collection, embeddings_dict):
    data = [
        list(embeddings_dict.keys()),  # Sentences
        list(embeddings_dict.values())  # Embeddings (numpy arrays)
    ]
    
    # Convert NumPy arrays to lists (Milvus requires lists)
    data[1] = [embedding.tolist() for embedding in data[1]]

    # Insert into Milvus
    collection.insert(data)
    collection.flush()
    print(f"Inserted {len(embeddings_dict)} embeddings into Milvus.")

# Insert embeddings
insert_embeddings(collection, embeddings_other_1)
insert_embeddings(collection, embeddings_other_2)


Inserted 4 embeddings into Milvus.
Inserted 3 embeddings into Milvus.


In [163]:
# Load the collection
collection_name = "sentence_embeddings"
collection = Collection(name=collection_name)
collection.load()

# Search parameters for Milvus (IVF_FLAT example)
search_params = {"metric_type": "L2", "params": {"nprobe": 10}}


# Dictionary to store flagged sentences
flagged_sentences = {}

# Compare input sentences with existing vector database
for input_sentence in embeddings_self:
    input_vector = embeddings_self[input_sentence].tolist()  # Convert to list for Milvus

    # Search in Milvus
    results = collection.search(
        data=[input_vector],
        anns_field="embedding",
        param=search_params,
        limit=1,  # Get the most similar vector
        output_fields=["sentence"]  # Retrieve stored sentence
    )

    # Extract similarity score
    max_similarity = 0
    if results and results[0]:  # Check if there are results
        for hit in results[0]:  # Milvus returns a list of results per query
            similarity = hit.distance  # Distance in COSINE metric is already similarity
            max_similarity = max(max_similarity, similarity)  # Track max similarity

    # Flag sentences with similarity >= 0.9
    flagged_sentences[input_sentence] = max_similarity <= 0.9

In [164]:
flagged_sentences


{'This is a plagarigized text.': True,
 'In finance, algorithms analyze vast datasets to predict market trends and assist in decision-making.': False,
 'Despite these benefits, concerns around AI ethics and bias remain critical topics of discussion.': False,
 'The ongoing development of AI promises both immense potential and significant challenges for society.': False}

In [165]:
# # Compare input sentences with existing files
# for other_file in existing_files:
#     for i, input_sentence in enumerate(embeddings_self):
#         max_similarity = 0
#         for output_sentence in other_file:
#             # Compute cosine similarity
#             similarity = cosine_similarity(
#                 [embeddings_self[input_sentence]],
#                 [other_file[output_sentence]]
#             )[0][0]  # Extract scalar value
#             if similarity > max_similarity:
#                 max_similarity = similarity
#         # Flag sentences with similarity >= 0.9
#         if max_similarity >= 0.9:
#             flagged_sentences[input_sentence] = True
#         else:
#             if input_sentence not in flagged_sentences:
#                 flagged_sentences[input_sentence] = False


In [166]:
# Output results
flagged_sentences, len(flagged_sentences)

({'This is a plagarigized text.': True,
  'In finance, algorithms analyze vast datasets to predict market trends and assist in decision-making.': False,
  'Despite these benefits, concerns around AI ethics and bias remain critical topics of discussion.': False,
  'The ongoing development of AI promises both immense potential and significant challenges for society.': False},
 4)

In [167]:
# Get only plagiarized sentences 
plagiarized_sentences = [sentence for sentence, flag in flagged_sentences.items() if flag]
plagiarized_sentences

['This is a plagarigized text.']

In [168]:
raise Exception("Stop here")

Exception: Stop here

# Highlighting PDF

In [ ]:
s1 = "This knowledge is also really important"
plagiarized_sentences.append(s1)
plagiarized_sentences

In [ ]:
import fitz
import string

In [ ]:


def highlight_plagiarized_text(pdf_path, plagiarized_sentences, output_path="highlighted.pdf"):
    """Highlights plagiarized sentences in a PDF, handling punctuation differences and multi-line sentences."""
    
    doc = fitz.open(pdf_path)
    
    for page in doc:
        # Extract list of words with their bounding rectangles
        words = page.get_text("words")
        if not words:
            continue
        
        # Prepare list of normalized words with their rectangles
        word_list = [
            (
                word[4].strip().strip(string.punctuation),  # Normalize: remove whitespace and punctuation
                fitz.Rect(word[:4])
            ) 
            for word in words
        ]
        
        for sentence in plagiarized_sentences:
            sentence = sentence.strip()
            if not sentence:
                continue
            
            # Normalize sentence words (remove punctuation and whitespace)
            sentence_words = [
                word.strip().strip(string.punctuation) 
                for word in sentence.split()
            ]
            sentence_words = [word for word in sentence_words if word]  # Remove empty strings
            
            if not sentence_words:
                continue
            
            # Check for matches in the page's words
            for i in range(len(word_list) - len(sentence_words) + 1):
                # Check if the sequence matches the normalized sentence words
                match = True
                for j in range(len(sentence_words)):
                    if word_list[i + j][0] != sentence_words[j]:
                        match = False
                        break
                if not match:
                    continue
                
                # Extract matched words and their rectangles
                matched_sequence = word_list[i:i + len(sentence_words)]
                if not matched_sequence:
                    continue
                
                # Group words into lines based on y0 coordinate
                line_groups = []
                current_line = [matched_sequence[0][1]]  # Start with the first word's rect
                current_y0 = matched_sequence[0][1].y0
                
                for k in range(1, len(matched_sequence)):
                    word_rect = matched_sequence[k][1]
                    # Check if the word is on the same line (using y0 within a small threshold)
                    if abs(word_rect.y0 - current_y0) < 1:
                        current_line.append(word_rect)
                    else:
                        line_groups.append(current_line)
                        current_line = [word_rect]
                        current_y0 = word_rect.y0
                line_groups.append(current_line)  # Add the last line
                
                # Highlight each line group separately
                for line in line_groups:
                    combined_rect = line[0]
                    for rect in line[1:]:
                        combined_rect |= rect  # Union of rectangles in the same line
                    page.add_highlight_annot(combined_rect)
    
    doc.save(output_path)
    print(f"✅ Highlighted PDF saved as: {output_path}")
    return output_path

In [ ]:
pdf_path = "test_files/22067311 Sardul Ojha.pdf"

output_pdf = highlight_plagiarized_text(pdf_path, plagiarized_sentences)

print(f"Highlighted PDF saved as: {output_pdf}")

✅ Highlighted PDF saved as: highlighted.pdf
Highlighted PDF saved as: highlighted.pdf


In [ ]:
raise ValueError("This is a test error message.")

ValueError: This is a test error message.

# Text Plagiarism to Vector Database

# Code Plagiarism

In [ ]:
file_path = "test_files/test.py"
code_file_name, code_file_content = next(iter(process_file(file_path).items()))
code_file_content

Successfully read Python file with utf-8 encoding.


'def print_hello_world():\n    print("Hello < -_- >")\n\nprint_hello_world()\na = 2\nb = 3\nprint(a+ b)'

In [ ]:
def process_code(input_string):
    # Split the string by newline, strip whitespace from each line and filter out empty lines
    return [line.strip() for line in input_string.split('\n') if line.strip()]

In [ ]:
process_code(code_file_content)

['def print_hello_world():',
 'print("Hello < -_- >")',
 'print_hello_world()',
 'a = 2',
 'b = 3',
 'print(a+ b)']

In [ ]:
import ast

# Read the Python file
file_path = "test_files/test.py"  # Change this to your Python file
with open(file_path, "r", encoding="utf-8") as f:
    source_code = f.read()

source_code = """
def calculate_stats(data):
    total = 0
    count = 0
    for value in data:
        total += value
        count += 1
    average = total / count if count != 0 else 0
    
    max_val = data[0]
    for num in data:
        if num > max_val:
            max_val = num
    
    return {
        'sum': total,
        'average': average,
        'max': max_val
    }

def filter_even(numbers):
    even = []
    for n in numbers:
        if n % 2 == 0:
            even.append(n)
    return even
"""

# Convert to AST
tree = ast.parse(source_code)

# Print the AST structure
print(ast.dump(tree, indent=4))

Module(
    body=[
        FunctionDef(
            name='calculate_stats',
            args=arguments(
                posonlyargs=[],
                args=[
                    arg(arg='data')],
                kwonlyargs=[],
                kw_defaults=[],
                defaults=[]),
            body=[
                Assign(
                    targets=[
                        Name(id='total', ctx=Store())],
                    value=Constant(value=0)),
                Assign(
                    targets=[
                        Name(id='count', ctx=Store())],
                    value=Constant(value=0)),
                For(
                    target=Name(id='value', ctx=Store()),
                    iter=Name(id='data', ctx=Load()),
                    body=[
                        AugAssign(
                            target=Name(id='total', ctx=Store()),
                            op=Add(),
                            value=Name(id='value', ctx=Load())),
               